In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


In [2]:
from langchain_openai import ChatOpenAI

PROVIDER = "groq"

LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini", 
    },
}
MAX_RETRIES = 5  


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed, 
        max_retries=MAX_RETRIES,
    )

llm = get_llm(temperature=0.7) 

In [3]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema  

response_schemas = [
    ResponseSchema(name="destination", description="여행지 (도시나 지역 이름)"),
    ResponseSchema(name="duration",    description="여행 기간 (예: 2박 3일). 언급이 없으면 '정보 없음'"),
    ResponseSchema(name="budget",      description="총 예산 또는 지출 금액 (예: 30만원). 언급이 없으면 '정보 없음'"),
    ResponseSchema(name="rating",      description="추천도 1~5 사이 숫자 하나. 점수 언급이 없으면 글의 만족도로 추정"),
    ResponseSchema(name="activities",  description="주요 활동 목록. 각 항목은 '장소 + 활동' 형태의 짧은 구",
                   type="List[string]"),   
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)

format_instructions = parser.get_format_instructions()
print(format_instructions)   

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"destination": string  // 여행지 (도시나 지역 이름)
	"duration": string  // 여행 기간 (예: 2박 3일). 언급이 없으면 '정보 없음'
	"budget": string  // 총 예산 또는 지출 금액 (예: 30만원). 언급이 없으면 '정보 없음'
	"rating": string  // 추천도 1~5 사이 숫자 하나. 점수 언급이 없으면 글의 만족도로 추정
	"activities": List[string]  // 주요 활동 목록. 각 항목은 '장소 + 활동' 형태의 짧은 구
}
```


In [4]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 여행 후기와 계획 글을 분석해 핵심 정보를 정리하는 전문가입니다.\n"
     "규칙:\n"
     "- 글에 명시된 내용을 우선으로 추출하고, 없는 정보는 지어내지 말고 '정보 없음'으로 쓰세요.\n"
     "- rating은 1~5 사이 숫자 하나만 쓰세요.\n"
     "- 반드시 아래 형식으로만 답하세요.\n\n"
     "{format_instructions}"),
    ("user", "여행 글: {text}"),
]).partial(format_instructions=format_instructions)

llm = get_llm(temperature=0)

chain = prompt | llm | parser

In [5]:
import json

text = "지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 해운대에서 바다구경하고, 자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. 정말 만족스러운 여행이었습니다. 5점 만점에 4점 정도 줄 수 있을 것 같아요."

result = chain.invoke({"text": text})

print(type(result).__name__)  
print(json.dumps(result, ensure_ascii=False, indent=4))

print(f"\n {result['destination']} |  {result['duration']} |  {result['budget']} |  {result['rating']}/5")
for act in result["activities"]:
    print(f"   - {act}")

dict
{
    "destination": "부산",
    "duration": "2박 3일",
    "budget": "30만원",
    "rating": "4",
    "activities": [
        "해운대 바다구경",
        "자갈치시장 회 먹기",
        "감천문화마을 구경"
    ]
}

 부산 |  2박 3일 |  30만원 |  4/5
   - 해운대 바다구경
   - 자갈치시장 회 먹기
   - 감천문화마을 구경


In [6]:
from langchain_core.exceptions import OutputParserException

texts = [
    text,
    "다음 달에 제주도로 3박 4일 여행 갈 계획이에요. 예산은 50만원 잡았고, 한라산 등반이랑 우도 자전거 일주, 흑돼지 맛집 투어를 하고 싶어요.",

    "강릉 다녀왔는데 비가 계속 와서 별로였어요. 안목해변 카페거리만 겨우 들렀네요. 2점 드립니다.",

    "도쿄 4박 5일 혼자 여행! 항공·숙박 포함 120만원 들었어요. 시부야 쇼핑, 아사쿠사 센소지 관광, 디즈니씨까지 알차게 다녔고 5점 만점!",
]

REQUIRED_KEYS = {"destination", "duration", "budget", "rating", "activities"}

results = chain.batch([{"text": t} for t in texts], return_exceptions=True)

for i, r in enumerate(results, 1):
    print(f"[{i}] ", end="")
    if isinstance(r, Exception):
        print(f"실패: {str(r).splitlines()[0]}")
        continue

    warnings = []
    if set(r) != REQUIRED_KEYS:
        warnings.append(f"키 불일치 {set(r) ^ REQUIRED_KEYS}")
    if not isinstance(r.get("activities"), list):
        warnings.append("activities가 리스트가 아님")
    if str(r.get("rating")).strip() not in {"1", "2", "3", "4", "5"}:
        warnings.append(f"rating 형식 이상: {r.get('rating')!r}")

    print("  " if not warnings else f" {warnings}")
    print("    ", json.dumps(r, ensure_ascii=False))

[1]   
     {"destination": "부산", "duration": "2박 3일", "budget": "30만원", "rating": "4", "activities": ["해운대 바다구경", "자갈치시장 회 먹기", "감천문화마을 구경"]}
[2]  ["rating 형식 이상: '정보 없음'"]
     {"destination": "제주도", "duration": "3박 4일", "budget": "50만원", "rating": "정보 없음", "activities": ["한라산 등반", "우도 자전거 일주", "제주 흑돼지 맛집 투어"]}
[3]   
     {"destination": "강릉", "duration": "정보 없음", "budget": "정보 없음", "rating": "2", "activities": ["안목해변 카페거리 방문"]}
[4]   
     {"destination": "도쿄", "duration": "4박 5일", "budget": "120만원", "rating": "5", "activities": ["시부야 쇼핑", "아사쿠사 센소지 관광", "디즈니씨 방문"]}
